# Chicago Crime Era Structural Analysis
## 

**Author:** Erik Pak<br>
**Date:** 04/2026<br>
**Data:** Chicago Data Portal - Crime Incidents 2001–2025

---
## Research Question
Do different crime types behave differently across Pre-COVID -> COVID -> Post-COVID, and is there a permanent structural shift away from the pre-COVID baseline?

## Era Definitions

| Era        | Period                  | Months |
|------------|-------------------------|--------|
| Pre-COVID  | Jan 2001 – Feb 2020     | 230    |
| COVID      | Mar 2020 – Dec 2022     | 34     |
| Post-COVID | Jan 2023 – Dec 2025     | 36     |

Era cutoff rationale: The COVID era begins in March 2020, coinciding with the Illinois stay-at-home order (March 21, 2020) and the WHO pandemic declaration (March 11, 2020). The post-COVID era begins in January 2023, following the expiration of Illinois's disaster proclamation and the effective end of major federal pandemic-era policies in late 2022. These boundaries are administrative and policy-based; the underlying behavioral and enforcement shifts may not align exactly with these dates.

In [1]:
import pandas as pd
import pyarrow as pa
import pyarrow.feather as feather
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
from scipy.spatial.distance import pdist, squareform
from scipy.spatial import procrustes
from scipy.stats import zscore
from sklearn.decomposition import PCA
from importlib.metadata import version

# python source path
sys.path.append('../Src/')

# seed
SEED = 1776

# custom python
import plot
import utils
import zscore_anomaly
import evaluate_clusters

# Create a dictionary of versions
versions = {
    "Python": sys.version.split()[0],
    "Pandas": pd.__version__,
    "NumPy": np.__version__,
    "Pyarrow": pa.__version__, 
    "Seaborn": sns.__version__,
    "Matplot": sys.modules['matplotlib'].__version__,
    "Sk-Learn": sys.modules['sklearn'].__version__,
    "Scipy": sys.modules['scipy'].__version__,
}

# Display as a clean DataFrame
df_versions = pd.DataFrame(list(versions.items()), columns=['Library', 'Version'])
print(df_versions)

# Remove scientific notation
np.set_printoptions(suppress=True, precision=4, linewidth=100)
# reset options
pd.set_option('display.max_rows', None)

# Use an int32 type instance to save memory
arrow_int32 = pd.ArrowDtype(pa.int32())

    Library Version
0    Python  3.13.9
1    Pandas   2.3.3
2     NumPy   2.3.4
3   Pyarrow  22.0.0
4   Seaborn  0.13.2
5   Matplot  3.10.7
6  Sk-Learn   1.8.0
7     Scipy  1.16.3


## Data Import
Chicago crime incident data was loaded from a PyArrow Feather file, and the dataset contains 8,469,443 records across 34 columns, spanning January 2001 through December 2025, after further processing by ChicagoCrimeEraAnalysis.ipynb Notebook.

In [2]:
# PyArrow's version of the 'arrow' backend
df = feather.read_feather('../Data/crime_data_covid.feather', memory_map=True, types_mapper=pd.ArrowDtype)
# display
df.head()

,case_number,date,block,iucr,primary_type,description,location_description,arrest,domestic,beat,...,day_of_week,quarter,year_quarter,time_of_day,fbi_code_desc,fbi_index_code,district_location,year_week,year_month,Indexed
0,01G050460,2001-01-24 20:45:00,072XX S RIDGELAND AV,1811,NARCOTICS,POSS: CANNABIS 30GMS OR LESS,SIDEWALK,True,False,0324,...,Wednesday,Q1,2001-Q1,Night,Drug Abuse Violations,False,Grand Crossing,2001-04,200101,N
1,03J493690,2003-07-12 17:00:00,0105XX S DOBSON AVE,0890,THEFT,FROM BUILDING,APARTMENT,False,False,0624,...,Saturday,Q3,2003-Q3,Evening,Larceny – Theft,True,Gresham,2003-28,200307,I
2,04X245238,2004-12-13 21:15:00,006XX N RIDGEWAY AVE,2024,NARCOTICS,POSS: HEROIN(WHITE),SIDEWALK,True,False,1122,...,Monday,Q4,2004-Q4,Night,Drug Abuse Violations,False,Harrison,2004-51,200412,N
3,07C115980,2006-03-31 09:15:00,026XX N NARRAGANSETT AVE,0610,BURGLARY,FORCIBLE ENTRY,APARTMENT,False,False,2512,...,Friday,Q1,2006-Q1,Morning,Burglary,True,Grand Central,2006-13,200603,I
4,07HN36467,2007-05-25 14:51:00,022XX N LA CROSSE AVE,1812,NARCOTICS,POSS: CANNABIS MORE THAN 30GMS,RESIDENCE,True,False,2522,...,Friday,Q2,2007-Q2,Afternoon,Drug Abuse Violations,False,Grand Central,2007-21,200705,N


In [3]:
df.info(verbose=True, show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8469443 entries, 0 to 8469442
Data columns (total 34 columns):
 #   Column                Non-Null Count    Dtype                                                       
---  ------                --------------    -----                                                       
 0   case_number           8469443 non-null  string[pyarrow]                                             
 1   date                  8469443 non-null  timestamp[s][pyarrow]                                       
 2   block                 8469443 non-null  string[pyarrow]                                             
 3   iucr                  8469443 non-null  string[pyarrow]                                             
 4   primary_type          8469443 non-null  string[pyarrow]                                             
 5   description           8469443 non-null  string[pyarrow]                                             
 6   location_description  8454105 non-

## The Timeline Definition
* To ensure the analysis is accurate, we define the three eras based on global lockdown patterns:
    * Pre-COVID: January 2001 – February 2020
    * COVID Era: March 2020 – December 31, 2022
    * Post-COVID: January 2023 – Present

In [4]:
df.era.unique()

<ArrowExtensionArray>
['pre_covid', 'post_covid', 'covid']
Length: 3, dtype: dictionary<values=string, indices=int8, ordered=0>[pyarrow]

In [5]:
# Spearate Era
df_pre = df[df.era == 'pre_covid']
df_covid = df[df.era == 'covid']
df_post = df[df.era == 'post_covid']
# display shape
df_pre.shape, df_covid.shape, df_post.shape

((7092647, 34), (623871, 34), (1052925, 34))

## Data Prep

In [6]:
# Time & Crime Type
df_pre = df_pre.groupby(['year_month', 'fbi_code_desc']).size()
df_pre.sort_index(inplace=True)
#
df_covid = df_covid.groupby(['year_month', 'fbi_code_desc']).size()
df_covid.sort_index(inplace=True)
#
df_post = df_post.groupby(['year_month', 'fbi_code_desc']).size()
df_post.sort_index(inplace=True)

# Returns a Series where the index is the level name and the value is the dtype
print(df_pre.index.dtypes)
# Multiindex name
print(df_pre.index.names, "\n")
print(df_pre.index.min(), df_pre.index.max())
print(df_covid.index.min(), df_covid.index.max())
print(df_post.index.min(), df_post.index.max(), "\n")
# convert this Series to a DataFrame
df_flat_pre = df_pre.reset_index(name="crime_count")
df_flat_covid = df_covid.reset_index(name="crime_count")
df_flat_post = df_post.reset_index(name="crime_count")
# Prevent Arrow Error
df_flat_pre.fbi_code_desc = df_flat_pre.fbi_code_desc.astype(str)
df_flat_covid.fbi_code_desc = df_flat_covid.fbi_code_desc.astype(str)
df_flat_post.fbi_code_desc = df_flat_post.fbi_code_desc.astype(str)

year_month                                         string[pyarrow]
fbi_code_desc    dictionary<values=string, indices=int32, order...
dtype: object
['year_month', 'fbi_code_desc'] 

('200101', 'Aggravated Assault') ('202002', 'Weapons Violations')
('202003', 'Aggravated Assault') ('202212', 'Weapons Violations')
('202301', 'Aggravated Assault') ('202512', 'Weapons Violations') 



In [7]:
df_flat_pre.nunique()

year_month        230
fbi_code_desc      26
crime_count      2431
dtype: int64

In [8]:
df_flat_covid.nunique()

year_month        34
fbi_code_desc     26
crime_count      535
dtype: int64

In [9]:
df_flat_post.nunique()

year_month        36
fbi_code_desc     26
crime_count      559
dtype: int64

## Normalization: Crimes Per Month

Raw crime counts cannot be compared directly across eras. Pre-COVID spans 230 months vs 34 (COVID) and 36 (Post-COVID). All comparisons use **crimes per month** to account for unequal era lengths. Era month counts are derived dynamically from the data, not hardcoded.

In [20]:
# Number of months per era: derive dynamically
era_months = (
    df.groupby('era')['year_month']
    .nunique()
    .to_dict()
)

era_dict = {
    'pre_covid':  (df_flat_pre),
    'covid':      (df_flat_covid),
    'post_covid': (df_flat_post),
}

# Update era_dict to include the month values
era_dict = {key: (df, era_months[key]) for key, df in era_dict.items()}
# era_dict = {key: (df, era_months.get(key)) for key, df in era_dict.items()} # guarded lookup

# # To get the DataFrame for COVID:
# df = era_dict['covid'][0]

# # To get the month count for COVID:
# months = era_dict['covid'][1]

# df, months = era_dict[key]

In [12]:
# Pivot (monthly crime counts for three years per era))
z_pivot_pre = matrix_flat_pre.pivot(index='fbi_code_desc', columns='year_month', values='crime_count').fillna(0)
z_pivot_covid = matrix_flat_covid.pivot(index='fbi_code_desc', columns='year_month', values='crime_count').fillna(0)
z_pivot_post = matrix_flat_post.pivot(index='fbi_code_desc', columns='year_month', values='crime_count').fillna(0)

# print shape
print("DataFrame Pre-Covid shape:", z_pivot_pre.shape)
print("DataFrame Covid shape:", z_pivot_covid.shape)
print("DataFrame Post-Covid shape:", z_pivot_post.shape, "\n")

DataFrame Pre-Covid shape: (26, 230)
DataFrame Covid shape: (26, 34)
DataFrame Post-Covid shape: (26, 36) 



In [10]:
# Aligns observations (rows)
common_idx = sorted(set(z_pivot_pre.index) &
                    set(z_pivot_covid.index) &
                    set(z_pivot_post.index))

# Re-index to ensure alignment (rows are crime type & columns are time)
z_pivot_pre   = z_pivot_pre.loc[common_idx].copy()
z_pivot_covid = z_pivot_covid.loc[common_idx].copy()
z_pivot_post  = z_pivot_post.loc[common_idx].copy()

#### Helper Function(s)

In [11]:
def process_crime_era(data: pd.DataFrame, metric: str='correlation'):
    """ 
    Standardizes rows, calculates correlation distance, and returns results.
    """
    # Vectorized & Explicit NaN handling & population std (ddof=0)
    z_scored = pd.DataFrame(
        zscore(data, axis=1, nan_policy='omit'),
        index=data.index,
        columns=data.columns
    )
    
    # Measure how dissimilar each pair of rows is based on their correlation (condensed distance vector)
    dist_vec = pdist(z_scored.values, metric=metric)
    
    # Reconstructs that into a full, symmetric distance matrix
    dist_df = pd.DataFrame(squareform(dist_vec), index=data.index, columns=data.index)

    # Print
    print(f"\nCondensed distance vector shape: {dist_vec.shape}")
    print(f"Symmetric distance matrix shape: {dist_df.shape}")
    print(f"Z-Score matrix shape: {z_scored.shape}")
    
    return dist_df, dist_vec, z_scored

In [12]:
dist_df_pre , dist_vec_pre, z_scored_pre = process_crime_era(z_pivot_pre)


Condensed distance vector shape: (325,)
Symmetric distance matrix shape: (26, 26)
Z-Score matrix shape: (26, 230)


## Compute the Correlation Distance Matrix 
* Correlation transforms a similarity measure into a dissimilarity metric, allowing distance-based methods to cluster variables by shared temporal patterns rather than absolute magnitude.
* $𝑑 = 1 - \text{corr}(x,y)$
* By default, pdist (and most distance functions in Scipy) treats rows as observations and columns as variables. It calculates the distance between the rows.

In [14]:
# Calculate Distance Vector & Matrix
dist_df_pre , dist_vec_pre, z_scored_pre = process_crime_era(z_pivot_pre)
dist_df_covid , dist_vec_covid, z_scored_pre = process_crime_era(z_pivot_covid)
dist_df_post , dist_vec_post, z_scored_pre = process_crime_era(z_pivot_post)


Condensed distance vector shape: (325,)
Symmetric distance matrix shape: (26, 26)
Z-Score matrix shape: (26, 230)

Condensed distance vector shape: (325,)
Symmetric distance matrix shape: (26, 26)
Z-Score matrix shape: (26, 34)

Condensed distance vector shape: (325,)
Symmetric distance matrix shape: (26, 26)
Z-Score matrix shape: (26, 36)


* Dark areas $\rightarrow$ high similarity (distance $\approx$ 0)
* Bright areas $\rightarrow$ low similarity (distance $\approx$ 2)
* Blocks along the diagonal $\rightarrow$ natural clusters

In [15]:
mats = [dist_df_pre, dist_df_covid, dist_df_post]
titles = ["Pre-Covid Correlation Distance (1 - corr)", "Covid Correlation Distance (1 - corr)", "Post-Covid Correlation Distance (1 - corr)"]

fig, axes = plt.subplots(1, 3, figsize=(32, 10))

heatmap_kwargs = {
    "cmap": "viridis",
    "xticklabels": True,
    "yticklabels": True,
    "annot": False,
    "fmt": ".2f",
    "cbar": True
}

for ax, mat, title in zip(axes, mats, titles):
    mask = np.triu(np.ones_like(mat, dtype=bool), k=0)   # mask upper triangle
    sns.heatmap(mat, ax=ax, mask=mask, **heatmap_kwargs)
    ax.set_title(title, fontsize=16, fontweight='bold')
    ax.tick_params(axis='x', rotation=45, labelsize=12)
    for label in ax.get_xticklabels():
        label.set_horizontalalignment('right')
    ax.tick_params(axis='y', labelsize=12)
    ax.set_xlabel("FBI Crime Type", fontsize=14)
    ax.set_ylabel("FBI Crime Type", fontsize=14)

plt.tight_layout()
plt.show()

## [PCA (Principal Component Analysis)](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html)
* PCA operates directly on the data matrix, via the covariance or correlation matrix, under a Euclidean geometry assumption.
    * Components are linear combinations of the original variables
    * How much of the total variance in the original data is captured by each component
    * High percentages are common because PCA is optimally reconstructive under Euclidean assumptions
## PCoA (Principal Coordinates Analysis):
* PCoA operates on a pairwise distance (or dissimilarity) matrix
    * May encode complex relationships (correlation, Bray–Curtis, etc.)
    * Coordinates are chosen to best preserve pairwise distances
    * Only positive eigenvalues (negative ones are noise/non-Euclidean artifacts)
## [Procrustes Analysis](https://docs.scipy.org/doc/scipy/reference/generated/scipy.spatial.procrustes.html)
* A statistical method for shape analysis used to compare two sets of data points (shapes)

In [16]:
def compare_pca_pcoa(data, dist_df, n_components=2):
    # PCA Calculation
    pca = PCA(n_components=n_components)
    coords_pca = pca.fit_transform(data.values)
    pca_var = pca.explained_variance_ratio_

    # PCoA Calculation
    D2 = dist_df.values ** 2
    n = D2.shape[0]
    J = np.eye(n) - np.ones((n, n)) / n
    B = -0.5 * J @ D2 @ J 

    eigvals, eigvecs = np.linalg.eigh(B)
    idx = np.argsort(eigvals)[::-1]
    eigvals, eigvecs = eigvals[idx], eigvecs[:, idx]

    coords_pcoa = eigvecs[:, :n_components] * np.sqrt(np.maximum(eigvals[:n_components], 0))
    total_pos_variance = np.sum(eigvals[eigvals > 0])
    pcoa_var = eigvals[:n_components] / total_pos_variance

    # Procrustes Alignment
    # mtx_pca and mtx_pcoa are now on the same scale/origin
    mtx_pca, mtx_pcoa, disparity = procrustes(coords_pca, coords_pcoa)

    # Display Output
    print(f"VARIANCE EXPLAINED ({n_components} Components)")
    print("PCA:  " + " & ".join([f"PC{i+1}={v:.2%}" for i, v in enumerate(pca_var)]))
    print("PCoA: " + " & ".join([f"Dim{i+1}={v:.2%}" for i, v in enumerate(pcoa_var)]))
    print(f"\nPROCRUSTES DISPARITY: {disparity:.4f}\n")

    # Plotting
    fig, axes = plt.subplots(1, 3, figsize=(24, 7))
    titles = ["PCA (Aligned)", "PCoA (Aligned)", "Procrustes Overlay"]
    embeddings = [mtx_pca, mtx_pcoa]

    # Plot individual Scatter plots
    for i in range(2):
        ax, emb = axes[i], embeddings[i]
        ax.scatter(emb[:,0], emb[:,1], c='skyblue', s=100, edgecolor='black', alpha=0.7)
        # for j, txt in enumerate(data.index):
        #     ax.annotate(txt, (emb[j,0], emb[j,1]), fontsize=8)
        ax.set_title(titles[i])

    # Plot Overlay
    ax_ov = axes[2]
    ax_ov.scatter(mtx_pca[:,0], mtx_pca[:,1], c='blue', label='PCA', alpha=0.5)
    ax_ov.scatter(mtx_pcoa[:,0], mtx_pcoa[:,1], c='red', label='PCoA', alpha=0.5)
    
    # Draw lines between corresponding points
    for i in range(len(mtx_pca)):
        ax_ov.plot([mtx_pca[i,0], mtx_pcoa[i,0]], 
                   [mtx_pca[i,1], mtx_pcoa[i,1]], 'gray', linestyle='--', alpha=0.3)
    
    ax_ov.set_title(f"{titles[2]} (Disparity: {disparity:.3f})")
    ax_ov.legend()

    plt.tight_layout()
    plt.show()

    return

In [17]:
compare_pca_pcoa(z_pivot_pre, dist_df_pre, n_components=2)

VARIANCE EXPLAINED (2 Components)
PCA:  PC1=97.89% & PC2=1.28%
PCoA: Dim1=55.62% & Dim2=19.46%

PROCRUSTES DISPARITY: 0.9154



#### Given a massive $\approx$0.94 disparity, our Correlation Distance (1 - r) tells a completely different story from PCA.
* PCA is misleading since it is only showing the volume of crime. To understand the actual shifting patterns of crime types, we must use PCoA because the underlying structure is fundamentally different.

## Compare Pre-COVID, COVID, and Post-COVID

In [18]:
# Calculate Distance Vector & Matrix
dist_df_pre , dist_vec_pre, z_score_pre = process_crime_era(z_pivot_pre)
dist_df_covid , dist_vec_covid, z_score_covid = process_crime_era(z_pivot_covid)
dist_df_post , dist_vec_post, z_score_post = process_crime_era(z_pivot_post)


Condensed distance vector shape: (325,)
Symmetric distance matrix shape: (26, 26)
Z-Score matrix shape: (26, 230)

Condensed distance vector shape: (325,)
Symmetric distance matrix shape: (26, 26)
Z-Score matrix shape: (26, 34)

Condensed distance vector shape: (325,)
Symmetric distance matrix shape: (26, 26)
Z-Score matrix shape: (26, 36)


In [19]:
from scipy.stats import kruskal, ranksums

# Statistical Testing (Omnibus)
h_stat, p_kruskal = kruskal(dist_vec_pre, dist_vec_covid, dist_vec_post)

# Post-hoc Testing (Pairwise)
# Using Bonferroni correction for 3 comparisons (alpha = 0.05 / 3 = 0.0167)
comparisons = [("Pre vs Covid", dist_vec_pre, dist_vec_covid), 
               ("Covid vs Post", dist_vec_covid, dist_vec_post), 
               ("Pre vs Post", dist_vec_pre, dist_vec_post)]

results = []
for label, v1, v2 in comparisons:
    stat, p_val = ranksums(v1, v2)
    results.append({
        "Comparison": label,
        "P-Value": round(p_val, 4),
        "Significant": p_val < 0.0167, # Adjusted Alpha: 0.05 / 3 = 0.01666
        "Median Diff": round(np.median(v2) - np.median(v1), 4)
    })

# Display Results
print(f"Kruskal-Wallis Test: H={h_stat:.4f}, p={p_kruskal:.4f}")
print("\nPairwise Comparisons:")
print(pd.DataFrame(results))

print("\nMedian Distances (Lower = More Synchronized):")
print(f"Pre: {np.median(dist_vec_pre):.4f} | Covid: {np.median(dist_vec_covid):.4f} | Post: {np.median(dist_vec_post):.4f}")

Kruskal-Wallis Test: H=49.1662, p=0.0000

Pairwise Comparisons:
      Comparison  P-Value  Significant  Median Diff
0   Pre vs Covid   0.0000         True       0.2418
1  Covid vs Post   0.0000         True      -0.1165
2    Pre vs Post   0.0008         True       0.1253

Median Distances (Lower = More Synchronized):
Pre: 0.5616 | Covid: 0.8034 | Post: 0.6869


#### The "Big Picture" Result (Omnibus)
* Kruskal-Wallis $p = 0.0374$. Since this is below $0.05$, we have statistically significant evidence that the three eras are not the same. The way crimes "move together" shifted measurably across these periods.

#### The Significant Shift: Pre vs. Covid
P-Value: $0.0127$ (Significant) | Median Diff: $-0.0146$ This is your most important finding.
* The Direction: The negative Median Diff means the distance decreased.
* The Interpretation: Crimes became more synchronized during COVID. The pandemic acted as a "systemic shock" that forced disparate crime types to follow a more similar trajectory (likely due to lockdowns and mobility restrictions affecting everything at once).

#### The "New Normal": 
* Covid vs. Post
    * P -Value: $0.2812$ (Not Significant).
    * This suggests that the "behavioral synchronization" that happened during COVID has not reverted to the Pre-COVID state yet. The relationship between crime during the "Post" period is statistically indistinguishable from that during the "Covid" period.

#### The Long-Term Trend
* Median Distances:
    * $0.7119 \rightarrow 0.6973 \rightarrow 0.6869$
    * Analysis of median correlation distances reveals a steady downward trend in dissimilarity ($0.7119 \rightarrow 0.6973 \rightarrow 0.6869$), suggesting that crime types have become increasingly synchronized over time. While the immediate shift from Pre-COVID to COVID was statistically significant ($p = 0.0127$), the long-term shift from Pre-COVID to Post-COVID ($p = 0.1145$) did not meet the strict Bonferroni-corrected threshold ($0.0167$), indicating that while a trend is visible, it has not yet reached a level of high statistical certainty.

## PLOT Era PCoA

In [20]:
def compare_to_pre_covid(dist_pre, dist_covid, dist_post, n_components=2):
    def get_pcoa(dist_df):
        # Traditional PCoA / Classical MDS
        D2 = dist_df.values ** 2
        n = D2.shape[0]
        J = np.eye(n) - np.ones((n, n)) / n
        B = -0.5 * J @ D2 @ J
        eigvals, eigvecs = np.linalg.eigh(B)
        idx = np.argsort(eigvals)[::-1]
        # Use maximum(0) to handle floating point noise in non-Euclidean distances
        coords = eigvecs[:, idx][:, :n_components] * np.sqrt(np.maximum(eigvals[idx][:n_components], 0))
        return coords

    # Generate Coordinates
    coords_pre = get_pcoa(dist_pre)
    coords_covid = get_pcoa(dist_covid)
    coords_post = get_pcoa(dist_post)

    # Procrustes Alignment (Pre-COVID is always the FIRST argument)
    # This standardizes Pre-COVID and aligns the others to it
    pre_std, covid_aligned, disp_covid = procrustes(coords_pre, coords_covid)
    _, post_aligned, disp_post = procrustes(coords_pre, coords_post)

    # Visualization
    fig, axes = plt.subplots(1, 2, figsize=(20, 8))
    
    eras = [
        (covid_aligned, "COVID", "red", axes[0], disp_covid),
        (post_aligned, "Post-COVID", "green", axes[1], disp_post)
    ]

    for aligned_data, name, color, ax, disp in eras:
        # Plot Baseline
        ax.scatter(pre_std[:, 0], pre_std[:, 1], c='blue', s=100, label='Pre-COVID (Baseline)', alpha=0.4, edgecolors='k')
        # Plot Target Era
        ax.scatter(aligned_data[:, 0], aligned_data[:, 1], c=color, s=100, label=f'{name} Era', alpha=0.8, edgecolors='k')
        
        # Draw "Stress Lines" (Vectors of change)
        for i in range(len(pre_std)):
            ax.plot([pre_std[i, 0], aligned_data[i, 0]], 
                    [pre_std[i, 1], aligned_data[i, 1]], 
                    'gray', linestyle='--', alpha=0.4)
            # Label the crimes on the baseline points
            ax.text(pre_std[i, 0], pre_std[i, 1], dist_pre.index[i], fontsize=9)

        ax.set_title(f"Shift: Pre-COVID --> {name}\nDisparity: {disp:.4f}", fontsize=14)
        ax.legend()
        ax.set_xlabel("PCoA Dim 1")
        ax.set_ylabel("PCoA Dim 2")

    plt.tight_layout()
    plt.show()

    return {
        "pre_std": pre_std, 
        "covid_aligned": covid_aligned, 
        "post_aligned": post_aligned,
        "disp_covid": disp_covid, 
        "disp_post": disp_post,
        "index_names": dist_pre.index
    }

In [21]:
# Plot
results = compare_to_pre_covid(dist_df_pre, dist_df_covid, dist_df_post, n_components=2)

#### In our Procrustes analysis, Disparity (often called $M^2$) measures the total lack of fit between two shapes. Since Pre-COVID, our 0.0 baseline (the "normal" state), these numbers represent the total structural distortion of the city's crime patterns. In Procrustes analysis, disparity measures how much you have to "stretch" or "bend" one era to make it look like the baseline.
  
#### The COVID "Shock": 0.6704
* In PCoA space, this means the eigenvectors (the underlying forces driving crime patterns) were rotated and stretched so severely that the "neighborhoods" of crime types were completely rearranged.

#### The Post-COVID "Partial Reset": 0.4287
* A drop from 0.67 to 0.43 ($\approx$ 36% decrease) is significant because it marks the transition from a state of chaos to a state of reorganization.
* This means that in the PCoA plot, the dots representing each crime are migrating back toward their Pre-COVID coordinates, even if they haven't arrived yet.

#### Value Range	
* 0.00 – 0.05	Almost no change in pattern.
    * Static / Anchored
* 0.06 – 0.15	Slight shift in relationship to other crimes.
    * Stable
* 0.16 – 0.22	Noticeable change
    * Volatile
* 0.23+	Radical departure from Pre-COVID
    * Regime Shift

In [22]:
def get_top_movers(pre_std, aligned_data, index, top_n=5):
    # Calculate Euclidean distance (residuals) since 
    # (pre_std and aligned_data) is in a physical location in a 2D space
    residuals = np.sqrt(np.sum((pre_std - aligned_data)**2, axis=1))
    
    # Create a Series for easy sorting
    mover_series = pd.Series(residuals, index=index)
    return mover_series.sort_values(ascending=False).head(top_n)

# Extract the variables from the results dictionary
pre_std = results["pre_std"]
covid_aligned = results["covid_aligned"]
post_aligned = results["post_aligned"]

# Usage:
covid_movers = get_top_movers(pre_std, covid_aligned, dist_df_pre.index)
post_movers = get_top_movers(pre_std, post_aligned, dist_df_pre.index)

print("TOP MOVERS (COVID):")
print(covid_movers)
print("\nTOP MOVERS (POST-COVID):")
print(post_movers)

TOP MOVERS (COVID):
fbi_code_desc
Fraud                           0.361119
Drug Abuse Violations           0.312985
Weapons Violations              0.301926
Homicide – 1st or 2nd Degree    0.232789
Disorderly Conduct              0.228248
dtype: float64

TOP MOVERS (POST-COVID):
fbi_code_desc
Fraud                                           0.517340
Weapons Violations                              0.267088
Disorderly Conduct                              0.262910
Involuntary Manslaughter / Reckless Homicide    0.261015
Drug Abuse Violations                           0.240382
dtype: float64


In [23]:
def get_stability_ranking(pre_std, aligned_data, index, top_n=5):
    # Calculate Euclidean distance (residuals) since 
    # (pre_std and aligned_data) is in a physical location in a 2D space
    residuals = np.sqrt(np.sum((pre_std - aligned_data)**2, axis=1))
    
    # Create Series: Lowest values = Most Stable
    stability_series = pd.Series(residuals, index=index)
    return stability_series.sort_values(ascending=True).head(top_n)

# Run it for Post-COVID
stable_crimes = get_stability_ranking(results["pre_std"], results["post_aligned"], results["index_names"])
print("MOST STABLE CRIMES (POST-COVID):\n", stable_crimes)

MOST STABLE CRIMES (POST-COVID):
 fbi_code_desc
Aggravated Battery    0.012166
Larceny – Theft       0.036062
Aggravated Assault    0.060280
Arson                 0.0671055
Simple Battery        0.068411
dtype: float64


## Save to File

In [24]:
# # Use a loop for NumPy saves
# arrays = {
#     'pre': dist_vec_pre,
#     'covid': dist_vec_covid,
#     'post': dist_vec_post
# }

# for era, data in arrays.items():
#     np.save(f'../Data/dist_vec_{era}.npy', data)

# # Optimize Feather saves
# # Feather requires string indexes to be columns. reset_index() ensures 
# dfs = {
#     'pre': dist_df_pre,
#     'covid': dist_df_covid,
#     'post': dist_df_post
# }

# for era, data in dfs.items():
#     # reset_index() moves the FBI descriptions into the data block
#     # compression='lz4' is the Feather default (super fast)
#     data.reset_index().to_feather(f"../Data/dist_df_{era}.feather")

# # crime counts
# counts = {'pre': z_pivot_pre,
#             'covid': z_pivot_covid,
#             'post': z_pivot_post
#            }

# for era, data in counts.items():
#     # reset_index() moves the FBI descriptions into the data block
#     # compression='lz4' is the Feather default (super fast)
#     data.to_feather(f"../Data/counts_{era}.feather")

# # DataFrame save
# df.reset_index().to_feather(f"../Data/crime.feather")